# Logit lens on Rohan's fine-tuned checkpoints

Runs the shared `xling` pipeline (logit lens, transfer status, layer profiles) on the two LoRA adapters from Rohan's `4610_colab_pilot.ipynb`. No training happens here.

| Checkpoint | What it is | Judged against | Trained answer |
|---|---|---|---|
| `base` | Qwen3-1.7B-Base | n/a | n/a |
| `round1` | Rohan's round-1 adapter (original fictional facts, English only) | `base` | old (round-1) answer |
| `round2` | Rohan's round-2 adapter (same LoRA, continued on the conflicting facts) | `round1` | new (round-2) answer |

**Data:** the 43-fact pilot Rohan trained on (`data/rohan_pilot_v6.2.json`), converted by `data/convert_rohan.py`. With `STYLE = "qa"` (default), every prompt is exactly Rohan's `Q: {question}\nA:`. The `norm_margin` column in `accuracy.csv` is then his forced-choice margin, and section 3 checks that we reproduce his numbers before trusting anything else.

**Two kinds of metric:**
- **Forced choice** (`norm_*`, Rohan's): mean log-prob per token of the whole answer, new vs old. This decides each row's status (`CRITERION = "norm"`).
- **Logit lens** (`D`, `lens.csv`): log p(new) − log p(old) for the answer's *first token*, read out at every layer. This gives the "where in depth" curves. Rows where old and new share a first token (e.g. *Lake Silver* / *Lake Amber*) have `lens_ok = False` and are left out of the curves.

**Runtime:** GPU (T4 or better). Evaluation takes a few minutes. Upload the `64610` folder to `MyDrive/64610`. If `adapters/rohan/` didn't come along (the two adapters are about 140 MB), cell 2 downloads them from Rohan's Dropbox.

In [ ]:
# The project folder (with xling/, data/, notebooks/, adapters/) lives in Google Drive.
from google.colab import drive
drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/64610"
%cd {PROJECT_DIR}
import sys
sys.path.insert(0, PROJECT_DIR)

In [ ]:
# Same library versions Rohan trained with.
!pip -q install "transformers==4.57.6" "peft==0.18.0"
# Colab preinstalls an old torchao that makes recent peft refuse to load; we don't use it.
!pip -q uninstall -y torchao

In [ ]:
BASE      = "Qwen/Qwen3-1.7B-Base"
STYLE     = "qa"          # "qa" = Rohan's exact prompts; "fewshot" = our 3-example prompts
CRITERION = "norm"        # status by Rohan's forced choice; "top1" / "full" also work
FACTS     = f"data/rohan_facts_{STYLE}.jsonl"
ADAPTERS  = ["adapters/rohan/round1", "adapters/rohan/round2"]   # round1 first: round2's reference
OUT       = f"results/rohan_{STYLE}"
EDIT, REF, STAGES = "round2", "round1", ("base", "round1", "round2")

## 1. Adapters and facts
Uses `adapters/rohan/round{1,2}` if they're already in the folder. Otherwise downloads Rohan's Dropbox folder of runs (about 1 GB, mostly optimizer states) and extracts only the two `*_best_adapter` weights. `convert_rohan.py` then writes the facts file and the adapters' `meta.json`.

In [ ]:
import os, zipfile, subprocess
from pathlib import Path

RUNS_URL = ("https://www.dropbox.com/scl/fo/lczrsrkodb51bx94l8nhj/AF7KBnSFQyjfzO2BKjpmFJ0"
            "?rlkey=piduo54rxaueay6xqirpqmorr&dl=1")
need = [p for p in ADAPTERS if not (Path(p) / "adapter_model.safetensors").exists()]
if need:
    if not os.path.exists("/content/rohan_runs.zip"):
        subprocess.run(["wget", "-q", "-O", "/content/rohan_runs.zip", RUNS_URL], check=True)
    with zipfile.ZipFile("/content/rohan_runs.zip") as z:
        for rnd in ("1", "2"):
            dest = Path(f"adapters/rohan/round{rnd}"); dest.mkdir(parents=True, exist_ok=True)
            for f in ("adapter_config.json", "adapter_model.safetensors"):
                (dest / f).write_bytes(z.read(f"round_{rnd}_best_adapter/{f}"))
    print("extracted adapters")

!python data/convert_rohan.py --style {STYLE}
for p in ADAPTERS:
    print(p, sorted(os.listdir(p)))

## 2. Load and validate
`validate_facts` lists the rows whose old and new answers share a first token. Those rows (8 in the `qa` style) still count for forced-choice accuracy but carry no lens signal. The tables show the logit lens on one prompt (deepest layer on top), before and after Rohan's round-2 adapter.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from xling.modeling import load_model, attach_adapter
from xling.schema import load_facts, validate_facts
from xling.plots import lens_grid

model, tok = load_model(BASE)
facts = load_facts(FACTS)
problems = validate_facts(facts, tok)
print(f"{len(facts)} facts; {len(problems)} rows without a usable first token:")
for p in problems: print("  ", p)

demo = facts[0].prompts["fr"][0]
print(repr(demo))
display(lens_grid(model, tok, demo, every=2))
pm = attach_adapter(model, ADAPTERS[1])
display(lens_grid(pm, tok, demo, every=2))
model = pm.unload()      # back to the plain base model

## 3. Evaluate
Scores `base` on all 43 facts, then each adapter. Writes `accuracy.csv`, `lens.csv`, `status.csv` and `run_info.json` to `OUT`.

In [ ]:
from xling.evaluate import run, summarize, transfer_status, failure_modes

acc, lens, status = run(FACTS, BASE, ADAPTERS, OUT, model=model, tok=tok, criterion=CRITERION)

### Reproduction check against Rohan's run
Before reading anything else: our `norm_margin` should match the `margin` column in Rohan's `*_answer_scores.csv` (copied into `data/rohan_reference/`). Expect small bf16 differences, more so on a different GPU than his L4. Sign flips should only happen for margins near 0. A large gap means the adapters didn't load as trained. Only meaningful for `STYLE = "qa"`.

In [ ]:
names = {"A_base": "base", "B_after_round1": "round1", "C_after_round2": "round2"}
ref = pd.concat([pd.read_csv(f"data/rohan_reference/{n}_answer_scores.csv") for n in names])
ref["model"] = ref.checkpoint.map(names)
ref["fact_id"] = ref.id.map(lambda i: f"fic_{i:02d}")
ref = ref.rename(columns={"language": "lang", "margin": "rohan_margin"})
cmp = acc.merge(ref[["model", "fact_id", "lang", "rohan_margin"]], on=["model", "fact_id", "lang"])
cmp["abs_diff"] = (cmp.norm_margin - cmp.rohan_margin).abs()
cmp["same_sign"] = (cmp.norm_margin > 0) == (cmp.rohan_margin > 0)
display(cmp.groupby("model").agg(rows=("abs_diff", "size"), max_abs_diff=("abs_diff", "max"),
                                 mean_abs_diff=("abs_diff", "mean"), same_sign=("same_sign", "mean")).round(3))
flips = cmp[~cmp.same_sign][["model", "fact_id", "lang", "norm_margin", "rohan_margin"]]
if len(flips): print("sign flips:"); display(flips.round(3))

# Rohan's headline table, recomputed from our numbers (his conditional_transfer_summary.csv below it)
m = acc[acc.model.isin(["round1", "round2"])].pivot_table(index=["fact_id", "lang"], columns="model", values="norm_margin")
en = m.xs("en", level="lang")
edited = en.index[(en.round1 < 0) & (en.round2 > 0)]
rows = [dict(language=l, english_edited_facts=len(edited),
             success=int((m.xs(l, level="lang").loc[edited].round2 > 0).sum())) for l in ["en", "fr", "zh", "fi"]]
display(pd.DataFrame(rows).assign(conditional_transfer=lambda d: (d.success / d.english_edited_facts).round(2)))
display(pd.read_csv("data/rohan_reference/conditional_transfer_summary.csv").round(2))

## 4. Transfer status
`summarize(status)` has one row per checkpoint and language:
- **`round1` vs `base`:** was the *original* fact learned in English, and did it reach each language?
- **`round2` vs `round1`:** was the update learned, and did it transfer? A `round2` row only counts when `round1` preferred the old answer *in that language* (otherwise it's `ref_unknown`). So `success_rate` here is the strict version of Rohan's conditional transfer: it asks whether the language actually *switched*, not whether it ended on the new answer.

`ref_already_correct` rows (the reference already preferred the trained answer before training) say little about learning. The second table drops them.

In [ ]:
print(f"criterion = {CRITERION}")
display(summarize(status))
print("without ref_already_correct rows:")
display(summarize(status[~status.ref_already_correct.astype(bool)]))

# Same tables with the first-token criterion, for comparison (no GPU needed: recomputed from the tables).
top1 = pd.concat([transfer_status(acc, lens, "round1", "base", "old", criterion="top1"),
                  transfer_status(acc, lens, "round2", "round1", "new", criterion="top1")])
print("criterion = top1 (first token is the model's top prediction):")
display(summarize(top1))

## 5. Logit lens figures
All figures leave out `lens_ok == False` rows. `save_all` also writes every figure as a PNG to `OUT/figures/` for the report.

In [ ]:
from xling import plots
lens_ok = lens[lens.lens_ok.astype(bool)]

plots.plot_stages(acc, STAGES, metric="norm"); plt.show()
plots.plot_D_by_layer(lens_ok, "round1", "base"); plt.show()
plots.plot_D_by_layer(lens_ok, EDIT, REF); plt.show()
plots.plot_delta_D(lens_ok, EDIT, REF); plt.show()

### Successful vs failed transfer, and where failures happen
The proposal's main comparison. Read the D curves for `transfer_ok` vs `transfer_fail` per language, then the peak-vs-final scatter:
- **Near the origin:** the update never reached that language (encoding).
- **On the diagonal but still wrong:** the old answer outweighs it (retrieval).
- **Well below the diagonal:** it was there mid-network and lost late (decoding).

The table counts these labels from `failure_modes`. With 10–25 facts per language these are anecdotes, not statistics.

In [ ]:
st = status[status.model == EDIT]
for make in (lambda: plots.plot_D_by_status(lens_ok, st, EDIT),
             lambda: plots.plot_profile(st, EDIT),
             lambda: plots.plot_onset(st, EDIT)):
    try:
        make(); plt.show()
    except ValueError as e:
        print("skipped:", e)

fm = failure_modes(st)
display(pd.crosstab(fm.lang, fm.failure_mode, margins=True))
display(fm[["fact_id", "lang", "peak_layer", "peak_dD", "final_dD", "final_D", "failure_mode"]]
        .sort_values(["lang", "failure_mode"]).round(2))

print(plots.save_all(acc, lens, status, OUT, EDIT, REF, STAGES))

## 6. Hand-off to activation patching
The patching cases are `round2`'s `transfer_fail` rows with a usable lens (`lens_ok`). Score patched runs with `xling.scoring.score_logprobs` so Δ_l is in the same units as `D` here.

In [ ]:
cases = st[(st.status == "transfer_fail") & st.lens_ok.astype(bool)][["model", "fact_id", "lang", "prompt_idx"]]
cases.to_csv(f"{OUT}/patch_cases.csv", index=False)
print(len(cases), "cases ->", f"{OUT}/patch_cases.csv")

## Caveats
- The French, Chinese and Finnish round-1 answers were drafted by Rohan and need native-speaker review.
- One test question per language per fact, so each status rests on a single prompt.
- Status uses the whole-answer forced choice; the lens uses the first token only. A row can be `transfer_fail` while its first-token `final_D` is slightly positive, or the other way round.
- In the `qa` style the Chinese answers follow `A:` with a space, as Rohan scored them. The first token of `" 绿色"` is a byte fragment shared by thousands of characters, so the lens reads the no-space form (`绿色`) instead. `STYLE = "fewshot"` avoids this, at the cost of prompts Rohan never evaluated on.